# Fake Bills Detection - Model Training & Evaluation

Binary classification: predict whether a bill is **genuine (1)** or **fake (0)** from six geometric measurements.
Only Python libraries (pandas, NumPy, scikit-learn, matplotlib) are used. Models are saved with `pickle`, not joblib.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

from src.utils import (
    load_data, split_features_target, evaluate_model,
    plot_confusion_matrix, plot_roc_curve, plot_feature_importance,
    plot_model_comparison, save_model, TARGET_COLUMN,
)

%matplotlib inline
RANDOM_STATE = 42
DATA_PATH = "../dataset/fake_bills.csv"
PLOTS_DIR = "../results/plots"
MODEL_PATH = "../results/models/best_model.pkl"

## 1. Load data

In [ ]:
df = load_data(DATA_PATH)
X, y = split_features_target(df)
print("Shape:", df.shape)
print("Missing values per column:\n", X.isna().sum())
print("Class balance:\n", y.value_counts())

## 2. Train / test split

Stratified 80/20 split so both sets keep the genuine/fake ratio. Imputation and scaling live **inside** the pipeline,
so they are fitted on training data only (no leakage).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)

## 3. Compare several models with 5-fold cross-validation

In [ ]:
def make_pipe(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

candidates = {
    "Logistic Regression": make_pipe(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "KNN": make_pipe(KNeighborsClassifier()),
    "SVM (RBF)": make_pipe(SVC(kernel="rbf")),
    "Random Forest": make_pipe(RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    "Gradient Boosting": make_pipe(GradientBoostingClassifier(random_state=RANDOM_STATE)),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    cv_results[name] = scores.mean()
    print(f"{name:<22} accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

plot_model_comparison(cv_results, save_path=f"{PLOTS_DIR}/model_comparison.png")
plt.show()

## 4. Tune the best model

In [ ]:
best_name = max(cv_results, key=cv_results.get)
print("Best model from cross-validation:", best_name)

param_grids = {
    "Logistic Regression": {"model__C": [0.01, 0.1, 1, 10, 100]},
    "KNN": {"model__n_neighbors": [3, 5, 7, 11, 15]},
    "SVM (RBF)": {"model__C": [0.1, 1, 10, 100], "model__gamma": ["scale", 0.01, 0.1]},
    "Random Forest": {"model__n_estimators": [100, 200, 400], "model__max_depth": [None, 5, 10]},
    "Gradient Boosting": {"model__n_estimators": [100, 200], "model__learning_rate": [0.05, 0.1], "model__max_depth": [2, 3]},
}

search = GridSearchCV(candidates[best_name], param_grids[best_name], cv=cv, scoring="accuracy", n_jobs=1)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print(f"Best CV accuracy: {search.best_score_:.4f}")
best_model = search.best_estimator_

## 5. Evaluate on the held-out test set

In [ ]:
y_pred = best_model.predict(X_test)
if hasattr(best_model, "predict_proba"):
    y_score = best_model.predict_proba(X_test)[:, 1]
else:  # e.g. SVC without probability estimates
    y_score = best_model.decision_function(X_test)

metrics = evaluate_model(y_test, y_pred, y_score)
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

print()
print(classification_report(y_test, y_pred, target_names=["Fake", "Genuine"]))

In [ ]:
plot_confusion_matrix(y_test, y_pred, save_path=f"{PLOTS_DIR}/confusion_matrix.png")
plt.show()
plot_roc_curve(y_test, y_score, save_path=f"{PLOTS_DIR}/roc_curve.png")
plt.show()

## 6. Which features matter?

In [ ]:
final_step = best_model.named_steps["model"]
feature_names = list(X.columns)

if hasattr(final_step, "feature_importances_"):
    importances = final_step.feature_importances_
    title = "Feature importance"
elif hasattr(final_step, "coef_"):
    importances = np.abs(final_step.coef_[0])
    title = "Feature importance (|coefficient|, scaled features)"
else:
    # models like KNN / SVM-RBF: use a model-agnostic permutation importance
    from sklearn.inspection import permutation_importance
    r = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=1)
    importances = r.importances_mean
    title = "Permutation importance"

plot_feature_importance(feature_names, importances, save_path=f"{PLOTS_DIR}/feature_importance.png", title=title)
plt.show()

## 7. Save the final model (pickle) and make a prediction

In [ ]:
save_model(best_model, MODEL_PATH)
print("Saved to", MODEL_PATH)

sample = X_test.iloc[[0]]
pred = best_model.predict(sample)[0]
print("Sample bill:\n", sample.to_string(index=False))
print("Prediction:", "Genuine" if pred == 1 else "Fake", "| actual:", "Genuine" if y_test.iloc[0] == 1 else "Fake")

## 8. Conclusion

*Fill in after running:* best model, test accuracy / F1 / ROC-AUC, the most important features, and any limitations
(e.g. small dataset, imputed `margin_low` values).